# Model Context Protocol (MCP) 实操任务

本笔记本包含以下实操内容：
1. 环境依赖配置。
2. 通过 `stdio` 连接外部 Time MCP 服务（查询本地时间）。
3. 通过 `http` 连接外部 Kiwi MCP 服务（查询航班）。
4. 使用 `FastMCP` 创建并部署一个本地自定义数学运算 MCP 服务。
5. 连接并调用本地自定义 MCP 服务。

In [1]:
# 1. 安装必要的环境依赖
# 注意：在 Jupyter 中运行 shell 命令需在开头加 "!"
# 使用 uv 安装到当前 Jupyter 内核使用的 Python 环境
import sys
!uv pip install --python {sys.executable} langchain-mcp-adapters fastmcp mcp-server-time

Using Python 3.13.14 environment at: D:\Langchain\jupyter\.venv
Checked 3 packages in 170ms


In [2]:
# ===== Windows + Jupyter 兼容修复（务必在创建 MCP Client 之前运行）=====
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
import mcp.client.stdio as _mcp_stdio
import mcp.os.win32.utilities as _mcp_win

# 1) MCP SDK 在 Windows 上会把 sys.stderr 交给 subprocess.Popen，
#    而 Jupyter 的 sys.stderr 没有 fileno()，导致 UnsupportedOperation: fileno。
#    这里让子进程直接继承 stderr，绕开该限制。
_original_create_windows_process = _mcp_win.create_windows_process

async def _create_windows_process_compat(command, args, env=None, errlog=None, cwd=None):
    return await _original_create_windows_process(command, args, env=env, errlog=None, cwd=cwd)

_mcp_win.create_windows_process = _create_windows_process_compat
_mcp_stdio.create_windows_process = _create_windows_process_compat

# 2) 定位并加载项目根目录的 .env（从当前目录向上查找）
#    修复原因：Windows 系统环境变量里残留了过期的 DEEPSEEK_API_KEY（尾号 0a6c），
#    DeepSeek 返回 401 AuthenticationError；.env 中的新 key（尾号 513c）才是有效的。
#    override=True 让 .env 的值优先生效。
_env_path = None
_p = Path.cwd()
for _ in range(5):
    if (_p / ".env").exists():
        _env_path = _p / ".env"
        break
    _p = _p.parent

if _env_path is not None:
    load_dotenv(dotenv_path=_env_path, override=True)
    print("已加载 .env:", _env_path)
else:
    print("警告: 未找到 .env，继续使用系统环境变量")

# 3) 配置 DeepSeek 的 OpenAI 兼容接口，供 create_agent("deepseek-chat", ...) 使用
os.environ["OPENAI_BASE_URL"] = "https://api.deepseek.com/v1"
_deepseek_key = os.environ.get("DEEPSEEK_API_KEY", "")
if _deepseek_key:
    os.environ["OPENAI_API_KEY"] = _deepseek_key
    # 只打印末 4 位，避免完整 key 泄露到笔记本输出中
    print("DeepSeek API Key 已配置（末4位: %s）" % _deepseek_key[-4:])
else:
    print("错误: 未找到 DEEPSEEK_API_KEY，请先在 .env 中配置后再运行本单元")

# 4) Kiwi MCP 服务绕过本地代理（修复: 本地代理会断开 streamable HTTP 连接）
_noproxy = os.environ.get("NO_PROXY", "")
if "mcp.kiwi.com" not in _noproxy:
    _noproxy = (_noproxy.strip(",") + ",mcp.kiwi.com").strip(",")
    os.environ["NO_PROXY"] = _noproxy
    os.environ["no_proxy"] = _noproxy


已加载 .env: D:\Langchain\jupyter\.env
DeepSeek API Key 已配置（末4位: 513c）


## 实操任务 1：连接外部 MCP 服务

### 1.1 通过 `stdio` 方式连接 Time MCP 服务
此方法会将 Time MCP 服务器脚本下载到本地，并作为子进程运行，不存在网络延迟。

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.messages import HumanMessage
# 注意：create_agent 可能是您课程或框架中的自定义辅助函数，请确保其定义在当前环境中可用
from langchain.agents import create_agent

# 1. 实例化并建立连接 (Time MCP)
#创建一个 MCP Client，并告诉它：我要连接一个叫 time 的 MCP Server，这个 Server 通过 stdio 启动（使用当前 Jupyter 环境的 Python 直接运行 mcp-server-time），并把时区设置为上海。
#MultiServerMCPClient 中的 MultiServer 表示：一个 Client 可以管理多个 MCP Server
client_time = MultiServerMCPClient(
    {
        # MCP Server 名称
        "time": {
            #MCP Client 与 MCP Server 之间通过标准输入/标准输出进行通信
            "transport": "stdio",
            # 启动 Time MCP 服务的命令（uvx 会解析到不兼容的 mcp 2.x，因此直接使用当前环境的 Python）
            "command": sys.executable,
            # 时区，传给uvx时区参数
            "args": [
                #以模块方式启动 mcp-server-time
                "-m", "mcp_server_time",
                "--local-timezone=Asia/Shanghai"
            ]
        }
    }
)

In [4]:
# 2. 获取 Time MCP 工具 (Jupyter 内部支持顶层 await)
# 从 Time MCP Server 中，把它提供的工具（Tools）获取到 Python 程序里
#这一块代码可以拆成
#client_time
#    ↓
#MCP Client
#    ↓
#get_tools()
#    ↓
#向 Time MCP Server 询问：
#“你有哪些工具可以使用？”
#    ↓
#返回工具列表
#    ↓
#tools_time
tools_time = await client_time.get_tools()

#遍历 Time MCP Server 返回的所有工具
print("--- 成功获取以下工具 ---")
for tool in tools_time:
    print(f"工具名称: {tool.name}")
    print(f"描述: {tool.description}")
    print("-" * 20)

--- 成功获取以下工具 ---
工具名称: get_current_time
描述: Get current time in a specific timezone
--------------------
工具名称: convert_time
描述: Convert time between timezones
--------------------


In [5]:
# 确保使用 .env 中的有效 key（兼容只运行本单元、未先运行环境配置单元的场景）
import os
from pathlib import Path
from dotenv import load_dotenv

_env_p = Path.cwd()
for _ in range(5):
    if (_env_p / ".env").exists():
        load_dotenv(dotenv_path=_env_p / ".env", override=True)
        break
    _env_p = _env_p.parent
os.environ["OPENAI_BASE_URL"] = "https://api.deepseek.com/v1"
os.environ["OPENAI_API_KEY"] = os.environ.get("DEEPSEEK_API_KEY", "")
print("使用 DeepSeek key（末4位: %s）" % os.environ["OPENAI_API_KEY"][-4:])
# Kiwi MCP 服务绕过本地代理（修复: 代理会断开 streamable HTTP 连接）
_noproxy = os.environ.get("NO_PROXY", "")
if "mcp.kiwi.com" not in _noproxy:
    _noproxy = (_noproxy.strip(",") + ",mcp.kiwi.com").strip(",")
    os.environ["NO_PROXY"] = _noproxy
    os.environ["no_proxy"] = _noproxy


# 3. 创建 Agent 并绑定 Time 工具
agent_time = create_agent("deepseek-chat", tools_time)

# 4. 调用 Agent
response_time = await agent_time.ainvoke({
    "messages": [HumanMessage("现在几点了？")]
})

# 5. 格式化输出结果
print("\n--- Agent 运行过程与结果 ---")
for message in response_time["messages"]:
    message.pretty_print()

使用 DeepSeek key（末4位: 513c）

--- Agent 运行过程与结果 ---
================================ Human Message =================================

现在几点了？
================================== Ai Message ==================================

我来帮你查询当前时间。
Tool Calls:
  get_current_time (call_00_IgN1qNCyRI1TO4UnGo3l7052)
 Call ID: call_00_IgN1qNCyRI1TO4UnGo3l7052
  Args:
    timezone: Asia/Shanghai
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '{\n  "timezone": "Asia/Shanghai",\n  "datetime": "2026-08-14T14:07:07+08:00",\n  "day_of_week": "Friday",\n  "is_dst": false\n}', 'id': 'lc_39ddb5b5-4391-4e9b-95c3-1b419bacd975'}]
================================== Ai Message ==================================

当前时间是 **2026年8月14日 星期五 下午2点07分**（北京时间，Asia/Shanghai）。

请问还需要查询其他时区的时间吗？


### 1.2 通过 `http` 方式连接 Kiwi Travel MCP
远程连接方式，需要设置 System Prompt 来约束模型输出匹配的 locale 参数。

In [6]:
#让 DeepSeek Agent 连接 Kiwi 的远程 MCP Server，获取航班查询工具，然后让 Agent 根据用户的自然语言需求，自动调用航班工具查询
# 1. 建立 HTTP 连接（创建一个 MCP 客户端，并告诉它：我要连接 Kiwi 提供的远程 MCP Server）
#MultiServerMCPClient(...) MCP 客户端管理器 负责连接一个或者多个MCP Server
client_kiwi = MultiServerMCPClient(
    {
        # MCP Server 命名名字叫做这个
        "kiwi-com-flight-search": {
            #通过 HTTP 协议与 MCP Server 通信
            "transport": "http",
            ## MCP Server 的网络地址
            "url": "https://mcp.kiwi.com"
        }
    }
)

In [7]:
# 2. 获取航班查询工具
# 向 Kiwi MCP Server 请求：你这里有哪些工具？然后把这些工具加载到 Python 中
# await 获取远程MCP Tools是一个异步网络操作
tools_kiwi = await client_kiwi.get_tools()

# 3. 配置针对 Kiwi 工具包的 Agent 并约束 locale
# 创建一个真正能够使用 Kiwi 工具的 Agent
agent_kiwi = create_agent(
    #配置大脑
    model="deepseek-chat",
      #使用Kiwi MCP Server提供的这些工具
    tools=tools_kiwi,
    #给Agent定义行为规则
    system_prompt="You are a travel agent. Help user find best flights. No follow up questions. If the user uses Chinese, use zh-cn as the locale value."
)

In [8]:
# 确保使用 .env 中的有效 key（兼容只运行本单元、未先运行环境配置单元的场景）
import os #用于操作环境变量
from pathlib import Path #用于操作文件路径
from dotenv import load_dotenv #把 .env 文件里的变量加载到 Python 的环境变量中

_env_p = Path.cwd()
for _ in range(5):
    if (_env_p / ".env").exists():
        load_dotenv(dotenv_path=_env_p / ".env", override=True)
        break
    _env_p = _env_p.parent
os.environ["OPENAI_BASE_URL"] = "https://api.deepseek.com/v1"
os.environ["OPENAI_API_KEY"] = os.environ.get("DEEPSEEK_API_KEY", "")
print("使用 DeepSeek key（末4位: %s）" % os.environ["OPENAI_API_KEY"][-4:])
# Kiwi MCP 服务绕过本地代理（修复: 代理会断开 streamable HTTP 连接）
_noproxy = os.environ.get("NO_PROXY", "")
if "mcp.kiwi.com" not in _noproxy:
    _noproxy = (_noproxy.strip(",") + ",mcp.kiwi.com").strip(",")
    os.environ["NO_PROXY"] = _noproxy
    os.environ["no_proxy"] = _noproxy


# 4. 执行测试 让Agent开始工作
# ainvoke作用：异步调用 Agent，让 Agent 执行一次完整任务
response_kiwi = await agent_kiwi.ainvoke({
    #用户问题
    "messages": [HumanMessage("查一下2026年9月15日晚上从北京飞杭州的航班。")]
})

# 5. 打印结果
for message in response_kiwi["messages"]:
    message.pretty_print()


使用 DeepSeek key（末4位: 513c）
================================ Human Message =================================

查一下2026年9月15日晚上从北京飞杭州的航班。
================================== Ai Message ==================================

好的，让我为您搜索2026年9月15日晚上从北京飞往杭州的航班。

让我先确定具体的航班信息，由于是"晚上"出发，我会设置较晚的出发时间范围。
Tool Calls:
  search-flight (call_00_zbo9OoEbH8DGUG54xFp93165)
 Call ID: call_00_zbo9OoEbH8DGUG54xFp93165
  Args:
    flyFrom: 北京
    flyTo: 杭州
    departureDate: 15/09/2026
    dtime_from: 18
    dtime_to: 23
    locale: zh-cn
================================= Tool Message =================================
Name: search-flight

[{'type': 'text', 'text': '{\n  "query": "北京 → 杭州 on 15/09/2026, 1 adult [dep 18:00–23:00]",\n  "currency": "EUR",\n  "passengers": {\n    "adults": 1,\n    "children": 0,\n    "infants": 0\n  },\n  "resultsCount": 15,\n  "itineraries": [\n    {\n      "id": "0ddc0f5650e70000bae880a8_0",\n      "price": 90.0,\n      "priceFormatted": "90 EUR",\n      "totalDurationSeconds": 7500

另外，有一个替代的国内航班查询MCP服务，地址是：https://mcp.variflight.com/

不过这个服务是收费的，而且价格比较贵，即便是优惠过后也需要0.1元/次。

使用步骤是：
- 注册账号
- 申请API Key
- 接入MCP

In [9]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# 连接Kiwi MCP服务器
client = MultiServerMCPClient(
    {
        "VariFlight-Aviation": {
            "transport": "http",
            "url": "https://ai.variflight.com/servers/aviation/mcp/?api_key=您在飞友AI开放平台申请的Key"
        }
    }
)

tools = await client.get_tools()

agent = create_agent(
    model="deepseek-chat",
    tools=tools,
    system_prompt="You are a travel agent. Help user find best flights. No follow up questions."
)

response = await agent.ainvoke({
    "messages": [HumanMessage("查一下2026年5月15日晚上从北京飞杭州的航班。")]
})

for message in response["messages"]:
    message.pretty_print()

是否读取到： True
长度： 35
后4位： 513c
API Base URL： https://api.deepseek.com/v1
若后4位仍是 0a6c（旧 key），请先重新运行上方的环境配置单元；应为 513c


自定义MCP服务

In [24]:
%%writefile math_mcp_server.py
# 此单元格执行后，会在当前目录下自动生成 "math_mcp_server.py" 文件
# 注意：%%writefile 只负责把下面的代码写入文件，不会执行这些代码。
# 这个 MCP Server 由后面的“连接本地自定义 Math MCP”单元以子进程方式自动启动，
# 不要在 Jupyter 里直接运行 mcp.run(...)，否则会报 "Already running asyncio" 错误。

# 这一整块代码的作用：创建了一个最简单的 MCP Server，并向外提供 3 个数学工具：
# 加法、乘法、平方根。其他 MCP Client / Agent 可以连接这个 Server，然后调用这些工具。

# FastMCP 理解成：帮助我们快速创建 MCP Server 的工具类
from fastmcp import FastMCP

# 初始化并命名 MCP 服务
mcp = FastMCP("Math")

# 注册加法工具
@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers"""
    return a + b

# 注册乘法工具
@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers"""
    return a * b

# 注册平方根计算工具
@mcp.tool()
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

# 启动 MCP Server，并使用 stdio 作为通信方式
if __name__ == "__main__":
    # 通过 `python math_mcp_server.py` 以子进程方式运行时 __file__ 存在，正常启动服务；
    # 若直接在 Jupyter 单元格里执行，则只提示、不启动，避免 "Already running asyncio" 报错。
    if "__file__" in globals():
        mcp.run(transport="stdio")
    else:
        print("提示: 正在 Jupyter 中直接执行，stdio MCP 服务由客户端单元自动以子进程方式启动，无需在此启动。")


Overwriting math_mcp_server.py


### 2.2 连接本地自定义的 Math MCP
我们在客户端的配置中将 `command` 指定为当前 Jupyter 环境的 Python，并将 `args` 指向刚才生成的 `math_mcp_server.py`，客户端会自动将其调起为子进程执行。

In [26]:
# 这块代码的作用：
# 连接 MCP Server → 获取工具 → 把工具交给 Agent → 让 Agent 调用工具完成任务# 确保使用 .env 中的有效 key（兼容只运行本单元、未先运行环境配置单元的场景）

#读取 .env 中保存的 API Key
import os
from pathlib import Path
from dotenv import load_dotenv

#寻找.env
_env_p = Path.cwd()
for _ in range(5):
    if (_env_p / ".env").exists():
        # 把 .env 文件里的变量加载到 Python 的环境变量中
        load_dotenv(dotenv_path=_env_p / ".env", override=True)
        break
    _env_p = _env_p.parent
os.environ["OPENAI_BASE_URL"] = "https://api.deepseek.com/v1"
os.environ["OPENAI_API_KEY"] = os.environ.get("DEEPSEEK_API_KEY", "")
print("使用 DeepSeek key（末4位: %s）" % os.environ["OPENAI_API_KEY"][-4:])
# Kiwi MCP 服务绕过本地代理（修复: 代理会断开 streamable HTTP 连接）
_noproxy = os.environ.get("NO_PROXY", "")
if "mcp.kiwi.com" not in _noproxy:
    _noproxy = (_noproxy.strip(",") + ",mcp.kiwi.com").strip(",")
    os.environ["NO_PROXY"] = _noproxy
    os.environ["no_proxy"] = _noproxy


# 1. 建立与本地 Math 服务的连接 创建 MCP Client
client_math = MultiServerMCPClient(
    {
        "math_service": {
            "transport": "stdio",
            "command": sys.executable,
            "args": ["math_mcp_server.py"]
        }
    }
)

# 2. 获取工具列表
# MCP Client 建立连接后，需要先了解 Server 提供了哪些 Tool，才能把这些 Tool 交给 Agent 使用。
tools_math = await client_math.get_tools()

# 3. 创建带有数学工具的 Agent
agent_math = create_agent(
    model="deepseek-chat",
    tools=tools_math,
    system_prompt="You are a helpful agent. You must use tools to answer math questions."
)

# 4. 执行混合数学运算测试
response_math = await agent_math.ainvoke({
    "messages": [HumanMessage("467和529的平方根之和是多少?")]
})

# 5. 打印计算轨迹和最终结果
for message in response_math["messages"]:
    message.pretty_print()

使用 DeepSeek key（末4位: 513c）
================================ Human Message =================================

467和529的平方根之和是多少?
================================== Ai Message ==================================

I need to calculate the square root of 467 and the square root of 529, then add them together.
Tool Calls:
  square_root (call_00_Gv6HkhenMPjTNXK0X3kD8417)
 Call ID: call_00_Gv6HkhenMPjTNXK0X3kD8417
  Args:
    x: 467
  square_root (call_01_FD2IJmuLMJcpqP34L2II0336)
 Call ID: call_01_FD2IJmuLMJcpqP34L2II0336
  Args:
    x: 529
================================= Tool Message =================================
Name: square_root

[{'type': 'text', 'text': '21.61018278497431', 'id': 'lc_9b9ecdc3-3e4a-46ff-8933-05294beb4a71'}]
================================= Tool Message =================================
Name: square_root

[{'type': 'text', 'text': '23.0', 'id': 'lc_94bf666a-add7-4300-9378-1af5e4155a20'}]
================================== Ai Message ==================================
